In [1]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor 
import matplotlib.pyplot as plt

print(torch.__version__)
print(torchvision.__version__)


#Getting a dataset:
#Fashion MNIST


train_data  = datasets.FashionMNIST(
    root= "data",
    train= True,
    download= True,
    transform = torchvision.transforms.ToTensor(),
    target_transform=None
)

test_data  = datasets.FashionMNIST(
    root= "data",
    train= False,
    download= True,
    transform = torchvision.transforms.ToTensor(),
    target_transform=None
)


len(train_data), len(test_data)


    

2.11.0
0.26.0


(60000, 10000)

In [30]:
image, label = train_data[0]
image.shape

classes = train_data.classes
classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [38]:

class FashionMNISTModelV2(nn.Module):

    def __init__(self, input_shape: int, hidden_units: int, output_shape: int ):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels= input_shape, out_channels= hidden_units, 
            kernel_size=3,
            stride=1,
            padding= 1
            ),

            nn.ReLU(),


            nn.Conv2d(in_channels= hidden_units,
                out_channels= hidden_units,
                kernel_size=3,
                stride =1,
                padding = 1),
            

            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels= hidden_units,
            out_channels=hidden_units, 
            kernel_size= 3,
            stride=1
            , padding=1),

            nn.ReLU(),

            nn.Conv2d(in_channels= hidden_units,
            kernel_size = 3, 
            stride=1,
            padding=1,
            out_channels=hidden_units),

            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)


        )

        # the two layers are feature extractors and the last layer wiull serve as a classifier
        # to calculate the shape of the last linear layer. we just take the product of the dimensions:
        # layer

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features= hidden_units * 49, out_features=output_shape,)
        )


    def forward(self, x):
        x = self.conv_block_1(x)
        print(x.shape , " is the shape of the conv1 layer")
        x = self.conv_block_2(x)
        print(x.shape, " is the shape of the conv2 layer")
        x = self.classifier(x)
        print(x.shape, "is the shape of the classifier layer")
        return


In [39]:
torch.manual_seed(42)

model2 = FashionMNISTModelV2(
    input_shape=1,
    hidden_units=10,
    output_shape=len(classes),
    
)

In [40]:
print(f"shape of the image : {image.shape}")
model2(image)



shape of the image : torch.Size([1, 28, 28])
torch.Size([10, 14, 14])  is the shape of the conv1 layer
torch.Size([10, 7, 7])  is the shape of the conv2 layer


RuntimeError: mat1 and mat2 shapes cannot be multiplied (10x49 and 490x10)

# Stepping through the NN. Conv2d


In [25]:
torch.manual_seed(42)

images = torch.randn(size = (32, 3, 64, 64))

test_image = images[0]
print(f" Image batch shape : {images.shape}")
print(f' Single image shape: {test_image.shape}')
print(f"test image: \n {test_image}")

 Image batch shape : torch.Size([32, 3, 64, 64])
 Single image shape: torch.Size([3, 64, 64])
test image: 
 tensor([[[ 1.9269,  1.4873,  0.9007,  ...,  1.8446, -1.1845,  1.3835],
         [ 1.4451,  0.8564,  2.2181,  ...,  0.3399,  0.7200,  0.4114],
         [ 1.9312,  1.0119, -1.4364,  ..., -0.5558,  0.7043,  0.7099],
         ...,
         [-0.5610, -0.4830,  0.4770,  ..., -0.2713, -0.9537, -0.6737],
         [ 0.3076, -0.1277,  0.0366,  ..., -2.0060,  0.2824, -0.8111],
         [-1.5486,  0.0485, -0.7712,  ..., -0.1403,  0.9416, -0.0118]],

        [[-0.5197,  1.8524,  1.8365,  ...,  0.8935, -1.5114, -0.8515],
         [ 2.0818,  1.0677, -1.4277,  ...,  1.6612, -2.6223, -0.4319],
         [-0.1010, -0.4388, -1.9775,  ...,  0.2106,  0.2536, -0.7318],
         ...,
         [ 0.2779,  0.7342, -0.3736,  ..., -0.4601,  0.1815,  0.1850],
         [ 0.7205, -0.2833,  0.0937,  ..., -0.1002, -2.3609,  2.2465],
         [-1.3242, -0.1973,  0.2920,  ...,  0.5409,  0.6940,  1.8563]],

        

In [26]:
conv_layer = nn.Conv2d(in_channels= 3,
            out_channels=10, 
            kernel_size=(3,3),
            stride=1,
            padding=1)

convout = conv_layer(test_image)

In [27]:
convout.shape

torch.Size([10, 64, 64])

In [28]:
maxpoollayer = nn.MaxPool2d(kernel_size=2)

testimagethroughmaxpool = maxpoollayer(convout)

In [29]:
testimagethroughmaxpool.shape

torch.Size([10, 32, 32])